# Bit-flip rate vs memory frequency, for `alpha_sq` = 2 to 6, `eps_p = 0.1`

The sweep of `notebooks/sweeps/sweep_w_a_detuned.ipynb`, repeated at five cat
sizes. It covers the same `shift_a` grid, `w_b = w_d`, `eps_p` and models, plus
one `compensated_shift` point per cat size. Read that notebook first for the
model (`build_ats_hamiltonian_interaction_detuned`, full-ATS nonlinearity):

    w_a = w_d / 2 - shift_a * D_a        w_b = w_d

**Run one `alpha_sq` at a time.** Each `alpha_sq` is its own cluster job,
started only once the previous one has finished. Inside a job the points fan
out as before, one task per point at `GPUS_PER_POINT` (0.5) GPU. A job therefore
takes at most `len(points) * 0.5` GPUs. Set `MAX_IN_FLIGHT` to cap that further.

**Resume.** Each `alpha_sq` is written to
`results/sweep_w_a_detuned_alpha{alpha_sq}.json` as soon as its job returns. On
a re-run, an `alpha_sq` whose file exists is loaded rather than recomputed
(`RERUN = False`). A dead kernel or a failed job therefore costs only the
`alpha_sq` in progress.

**What changes with `alpha_sq`: truncation and tolerances only.** They follow
the ladder of `notebooks/sweeps/sweep_alpha_compensated.ipynb`:

| `alpha_sq` | `n_a` / `n_b` | `m` / `k` | loop `rtol`/`atol` | final `rtol`/`atol` | `res_tol` | final block |
|---|---|---|---|---|---|---|
| 2, 3    | 25 / 12 | 40 / 10 | `1e-8` / `1e-9`  | `1e-10` / `1e-11` | `1e-9`  | 20 |
| 4, 5, 6 | 35 / 16 | 50 / 12 | `1e-9` / `1e-10` | `1e-11` / `1e-12` | `1e-10` | 50 |

The first row is exactly the setting of `sweep_w_a_detuned.ipynb`. That
truncation is already at the edge at `alpha_sq = 3` (top-3 tail up to 9e-12
against the 1e-11 threshold), and the rate falls to ~1e-9 by `alpha_sq = 6`. So
the larger cats need the second row. The truncation check at the end is run at
every point.

## Running it

1. **Kernel**: the workspace pixi environment,
   `/workspace/.pixi/envs/default/bin/python` (**Python 3.13.x**, `ray 2.56.0`).
   These must match the cluster.
2. **Vault session**, once (lasts about a week):

   ```bash
   export VAULT_ADDR=https://vault.int.alice-bob.com
   vault login -method=oidc role=research_core_user
   ```

3. Run all cells. The sweep cell blocks until the last `alpha_sq` is done.

If you have already run something that imports `ray` in this kernel, restart it
first. Ray caches `RAY_AUTH_MODE` at first import.

In [ ]:
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import numpy as np

from anb_compute import ray as acr

import pathlib


def _find_repo():
    """Locate the repo checkout without hardcoding an absolute path.

    Notebooks have no __file__, so search upward from the working directory --
    which VS Code sets to either the notebook's folder or the workspace root,
    depending on `jupyter.notebookFileRoot`. Both are handled.
    """
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "floquet_lindblad").is_dir():
            return base
        for hit in sorted(base.glob("*/src/floquet_lindblad")):
            return hit.parents[1]
    raise RuntimeError(f"no checkout with src/floquet_lindblad found above {here}")


REPO = _find_repo()

# Shipped by PATH, not by name: a bare module name that resolves to a
# non-editable site-packages install ships a frozen copy, and local edits would
# never reach the worker. These prepend to the worker's sys.path, so the
# dynamiqs here shadows the image's stock 0.3.6 (which lacks `mesolve_fast`).
PKG = str(REPO / "src" / "floquet_lindblad")
# The patched dynamiqs checkout, expected beside the repo; $DYNAMIQS_SRC wins.
DYNAMIQS = os.environ.get("DYNAMIQS_SRC") or str(
    REPO.parent / "dynamiqs" / "dynamiqs"
)
for _p in (PKG, DYNAMIQS):
    if not pathlib.Path(_p).is_dir():
        raise RuntimeError(f"not a directory, cannot ship to workers: {_p}")

RUNTIME_ENV = acr.build_runtime_env(py_modules=[PKG, DYNAMIQS])

print("ready")

### The points

The `shift_a` grid is the one in `sweep_w_a_detuned.ipynb`. It must contain
`0` (the uncompensated point, the builder check against
`results/sweep_ref.json`) and `1` (the compensated `w_a`, compared with
`compensated_shift`).

In [ ]:
import numpy as np

ALPHA_SQ_LIST = [2.0, 3.0, 4.0, 5.0, 6.0]
EPS_P = 0.02

# Same grid as sweep_w_a_detuned.ipynb. w_a = Omega - shift_a * D_a:
# shift_a = 0 -> Omega (uncompensated), shift_a = 1 -> Omega - D_a (compensated).
SHIFT_A_LIST = sorted({*np.linspace(1.5, 0.0, 20).tolist(),
                       *np.linspace(1.1, 0.85, 10).tolist(), 1.0}, reverse=True)
SHIFT_B_SWEEP = 0.0  # bare buffer at w_d along the sweep
assert any(np.isclose(s, 0.0) for s in SHIFT_A_LIST) and any(
    np.isclose(s, 1.0) for s in SHIFT_A_LIST), "keep shift_a = 0 and 1 in the grid"

POINTS = (
    [dict(label=f"sweep {i}", model="detuned", shift_a=float(s), shift_b=SHIFT_B_SWEEP)
     for i, s in enumerate(SHIFT_A_LIST)]
    + [dict(label="compensated_shift", model="compensated_shift")]
)

# Cluster footprint: one alpha_sq at a time; inside it, one task per point.
GPUS_PER_POINT = 1
MAX_IN_FLIGHT = None  # None: all points of an alpha_sq at once; an int caps it
RERUN = False         # False: an alpha_sq with a results file is loaded, not rerun

# Truncation and tolerances: the bands of the ladder in
# sweep_alpha_compensated.ipynb that cover alpha_sq = 2..6 (first band whose
# alpha_sq_max is above the point). The first is sweep_w_a_detuned.ipynb's setting.
SCHEDULE = [
    dict(alpha_sq_max=3.5, n_a=25, n_b=12, m=40, k=10,
         rtol_loop=1e-8, atol_loop=1e-9, res_tol_loop=1e-7,
         rtol_final=1e-10, atol_final=1e-11, res_tol=1e-9,
         n_blocks_final=20, max_cycles=30),
    dict(alpha_sq_max=6.5, n_a=35, n_b=16, m=50, k=12,
         rtol_loop=1e-9, atol_loop=1e-10, res_tol_loop=1e-8,
         rtol_final=1e-11, atol_final=1e-12, res_tol=1e-10,
         n_blocks_final=50, max_cycles=35),
]


def point_settings(alpha_sq):
    """Truncation + tolerances for one alpha_sq: the first band it falls in."""
    band = next((b for b in SCHEDULE if alpha_sq < b["alpha_sq_max"]), None)
    if band is None:
        raise ValueError(f"alpha_sq = {alpha_sq} is past the ladder (< {SCHEDULE[-1]['alpha_sq_max']})")
    return {k: v for k, v in band.items() if k != "alpha_sq_max"}


def out_path(alpha_sq):
    return REPO / "results" / f"sweep_w_a_detuned_alpha{alpha_sq:g}_eps_p_0_02.json"


print(f"eps_p = {EPS_P}, {len(SHIFT_A_LIST)} shift_a values + compensated_shift "
      f"= {len(POINTS)} points per alpha_sq, {GPUS_PER_POINT} GPU each "
      f"(at most {GPUS_PER_POINT * (MAX_IN_FLIGHT or len(POINTS)):g} GPUs at a time)\n")
print(f"{'alpha_sq':>8} {'n_a':>4} {'n_b':>4} {'m':>4} {'k':>3} {'rtol_loop':>10} "
      f"{'rtol_final':>11} {'res_tol':>8} {'blocks':>7}  status")
for a in ALPHA_SQ_LIST:
    s = point_settings(a)
    status = "saved, will be loaded" if out_path(a).exists() and not RERUN else "to run"
    print(f"{a:8g} {s['n_a']:4d} {s['n_b']:4d} {s['m']:4d} {s['k']:3d} "
          f"{s['rtol_loop']:10.0e} {s['rtol_final']:11.0e} {s['res_tol']:8.0e} "
          f"{s['n_blocks_final']:7d}  {status}")

### One point, on one GPU worker

The same worker as in `sweep_w_a_detuned.ipynb`.

In [ ]:
def solve_point(*, label, model, shift_a=1.0, shift_b=1.0, **kwargs):
    """`floquet_lindblad.pipeline.solve_point_ks_safe` on one model, on the worker.

    `model="compensated_shift"` runs the registry builder as it is;
    `model="detuned"` runs `build_ats_hamiltonian_interaction_detuned` at the
    given shifts. The label and the shifts are copied onto the result (and onto
    an error entry, which has no `params`).
    """
    from floquet_lindblad.pipeline import solve_point_ks_safe as solve

    if model == "compensated_shift":
        res = solve(hamiltonian="compensated_shift", **kwargs)
    elif model == "detuned":
        from floquet_lindblad.models import Lindbladian
        from floquet_lindblad.models.ats import (
            build_ats_hamiltonian_interaction_detuned as build,
        )

        def detuned(**kw):
            H, jump_ops, jump_ops_LdL, output_phase, V, T_block, params = build(
                shift_a=shift_a, shift_b=shift_b, **kw
            )
            return Lindbladian(H, jump_ops, T_block, params,
                               jump_ops_LdL, output_phase, V)

        detuned.__name__ = "detuned"
        res = solve(hamiltonian=detuned, **kwargs)
    else:
        raise ValueError(f"unknown model {model!r}")

    res.update(label=label, model=model, shift_a=float(shift_a), shift_b=float(shift_b))
    return res

### The sweeps, one `alpha_sq` after the other

`sweep` is the coordinator of `sweep_w_a_detuned.ipynb`, with an optional cap on
the number of points in flight. The loop below runs it as one `acr.run` job per
`alpha_sq` and waits for each job to finish before starting the next. It saves
every job's results as soon as they return. A job that fails as a whole is
reported and skipped; re-run the cell to retry it.

In [ ]:
import json
import time


def sweep(worker, points, common, gpus_per_point=0.5, max_in_flight=None):
    """Coordinator: one GPU task per point, at most `max_in_flight` at a time.

    Runs as the driver on a worker; each task asks for its own GPU share.
    `worker` is passed in because it is defined in this notebook and ships by
    value.
    """
    import ray

    from anb_compute import ray as acr

    queue = list(points)
    refs = {}

    def submit_next():
        p = queue.pop(0)
        refs[acr.submit(worker, num_gpus=gpus_per_point, num_cpus=2, **common, **p)] = p

    for _ in range(len(queue) if max_in_flight is None else min(max_in_flight, len(queue))):
        submit_next()

    out = []
    while refs:
        ready, _ = ray.wait(list(refs), num_returns=1)
        p = refs.pop(ready[0])
        try:
            res = ray.get(ready[0])
        except Exception as exc:  # a task lost to infra, not to numerics
            res = {**p, "error": f"{type(exc).__name__}: {exc}"}
        out.append(res)
        if queue:
            submit_next()
        print(f"alpha_sq = {common['alpha_sq']:g}  {p['label']:<18} "
              + (res["error"] if "error" in res
                 else f"rate_bf = {res['rate_bf'].real:.6e}  "
                      f"res = {res['res_rel']:.1e}  "
                      f"apps = {res['n_apply']:d}"
                      + ("" if res["converged"] else "  (NOT converged)")))
    order = {p["label"]: i for i, p in enumerate(points)}
    out.sort(key=lambda d: order[d["label"]])
    return out


def to_jsonable(obj):
    """numpy-only copy of floquet_lindblad.io.to_jsonable (that module imports jax,
    which this client does not have). Complex -> {"real", "imag"}, arrays -> lists."""
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return to_jsonable(obj.tolist())
    if isinstance(obj, (np.complexfloating, complex)):
        return {"real": float(obj.real), "imag": float(obj.imag)}
    if isinstance(obj, (np.floating, float)):
        return float(obj)
    if isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    if isinstance(obj, (np.integer, int)):
        return int(obj)
    return obj


def _complex_hook(d):
    """{"real", "imag"} -> complex, as floquet_lindblad.io.from_jsonable does."""
    return complex(d["real"], d["imag"]) if d.keys() == {"real", "imag"} else d


results = {}  # alpha_sq -> list of point results
for alpha_sq in ALPHA_SQ_LIST:
    path = out_path(alpha_sq)
    if path.exists() and not RERUN:
        with open(path) as f:
            results[alpha_sq] = json.load(f, object_hook=_complex_hook)
        print(f"alpha_sq = {alpha_sq:g}: loaded {len(results[alpha_sq])} points from {path.name}")
        continue

    common = dict(alpha_sq=alpha_sq, eps_p=EPS_P, seed=0, **point_settings(alpha_sq))
    print(f"\n=== alpha_sq = {alpha_sq:g}: {len(POINTS)} points, "
          f"n_a x n_b = {common['n_a']} x {common['n_b']} ===")
    t0 = time.time()
    try:
        res = acr.run(
            sweep, solve_point, POINTS, common, GPUS_PER_POINT, MAX_IN_FLIGHT,
            runtime_env=RUNTIME_ENV,
            poll_seconds=5,
            # No resources here on purpose: the fan-out inside asks for its own.
        )
    except Exception as exc:  # the whole job failed: report, go on to the next alpha_sq
        print(f"alpha_sq = {alpha_sq:g}: job FAILED ({type(exc).__name__}: {exc}); "
              "re-run this cell to retry it")
        continue
    path.parent.mkdir(exist_ok=True)
    with open(path, "w") as f:
        json.dump(to_jsonable(res), f, indent=2)
    # Store the round-tripped copy, so fresh and loaded alpha_sq look the same.
    with open(path) as f:
        results[alpha_sq] = json.load(f, object_hook=_complex_hook)
    print(f"alpha_sq = {alpha_sq:g}: {time.time() - t0:.0f} s incl. job startup, "
          f"wrote {len(res)} points to {path.name}")

missing = [a for a in ALPHA_SQ_LIST if a not in results]
print("\nall alpha_sq done" if not missing else f"\nmissing alpha_sq: {missing}")

### Summary per `alpha_sq`

Frequencies are angular, in the units of `ats.W_A`; offsets are in MHz,
`(w_a - w_d/2) / 2pi`. Per `alpha_sq`:

- **check**: the sweep at `shift_a = 0` against `results/sweep_ref.json`, the
  same Hamiltonian (relative deviation; ~1e-4 or below between runs);
- **buffer**: the sweep at `shift_a = 1` (buffer at `w_d`) over
  `compensated_shift` (buffer at `w_d - D_b`), i.e. the buffer compensation on
  its own;
- **minimum**: the lowest rate along the sweep, where it sits, and how it
  compares with `compensated_shift` and with the uncompensated point.

In [ ]:
TO_MHZ = 1e3 / (2 * np.pi)


def dw_a(r):
    """w_a - w_d/2 of a point, in MHz."""
    p = r["params"]
    return (p["w_a"] - p["w_d"] / 2) * TO_MHZ


def cat_size(r):
    return r["params"]["epsilon_d"] / (2 * r["params"]["g"])


with open(REPO / "results" / "sweep_ref.json") as f:
    REF = [r for r in json.load(f, object_hook=_complex_hook)
           if "error" not in r and np.isclose(r["eps_p"], EPS_P)]

summary = {}
for alpha_sq, res in results.items():
    ok = [r for r in res if "error" not in r]
    sweep_pts = sorted((r for r in ok if r["label"].startswith("sweep")), key=dw_a)
    by_shift = lambda s: next((r for r in sweep_pts if np.isclose(r["shift_a"], s)), None)
    comp = next((r for r in ok if r["label"] == "compensated_shift"), None)
    ref = next((r for r in REF if np.isclose(cat_size(r), alpha_sq)), None)
    rate = np.array([r["rate_bf"].real for r in sweep_pts])
    i_min = int(np.argmin(rate)) if len(rate) else None
    summary[alpha_sq] = dict(
        n_ok=len(ok), n_bad=len(res) - len(ok),
        n_conv=sum(r["converged"] for r in ok),
        dwa=np.array([dw_a(r) for r in sweep_pts]), rate=rate,
        unc=by_shift(0.0), at_one=by_shift(1.0), comp=comp, ref=ref,
        min=sweep_pts[i_min] if i_min is not None else None,
        D_a=ok[0]["params"]["D_a"] if ok else np.nan,
    )

rate_of = lambda r: r["rate_bf"].real if r is not None else np.nan
print(f"{'alpha_sq':>8} {'ok/conv/fail':>13} {'uncomp':>11} {'check':>8} "
      f"{'shift_a=1':>11} {'compensated':>11} {'buffer':>7} "
      f"{'minimum':>11} {'at shift_a':>10} {'dw_a MHz':>9} {'min/comp':>9} {'min/unc':>8}")
for a, s in summary.items():
    check = (abs(rate_of(s["unc"]) / rate_of(s["ref"]) - 1)
             if s["unc"] is not None and s["ref"] is not None else np.nan)
    m = s["min"]
    print(f"{a:8g} {s['n_ok']:5d}/{s['n_conv']:2d}/{s['n_bad']:2d} "
          f"{rate_of(s['unc']):11.3e} {check:8.1e} "
          f"{rate_of(s['at_one']):11.3e} {rate_of(s['comp']):11.3e} "
          f"{rate_of(s['at_one']) / rate_of(s['comp']):7.3f} "
          f"{rate_of(m):11.3e} {m['shift_a'] if m else np.nan:10.3f} "
          f"{dw_a(m) if m else np.nan:9.4f} "
          f"{rate_of(m) / rate_of(s['comp']):9.3f} {rate_of(m) / rate_of(s['unc']):8.3f}")
for a, res in results.items():
    for r in res:
        if "error" in r:
            print(f"alpha_sq = {a:g}  {r.get('label')}: FAILED: {r['error']}")
print(f"\nD_a/2pi = {summary[next(iter(summary))]['D_a'] * TO_MHZ:+.4f} MHz "
      "(independent of alpha_sq). The minimum is only resolved to the grid spacing.")

### Rate vs memory detuning, all cat sizes

Left: the rates themselves. Right: each sweep divided by the `compensated_shift`
rate at the same `alpha_sq`, so the shapes can be compared across the five
decades the rate spans. Darker means a larger cat.

In [ ]:
import matplotlib.pyplot as plt

# Sequential blue ramp (one hue, light -> dark with alpha_sq), from the dataviz palette.
RAMP = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]
colors = {a: RAMP[i * (len(RAMP) - 1) // max(1, len(summary) - 1)]
          for i, a in enumerate(summary)}
D_a_mhz = summary[next(iter(summary))]["D_a"] * TO_MHZ

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.2), constrained_layout=True)
for a, s in summary.items():
    c = colors[a]
    axes[0].plot(s["dwa"], s["rate"], ".-", ms=5, lw=1.1, color=c,
                 label=rf"$|\alpha|^2 = {a:g}$")
    if s["comp"] is not None:
        axes[0].scatter([dw_a(s["comp"])], [rate_of(s["comp"])], s=60, marker="*",
                        color=c, edgecolors="white", linewidths=0.5, zorder=4)
        axes[1].plot(s["dwa"], s["rate"] / rate_of(s["comp"]), ".-", ms=5, lw=1.1,
                     color=c, label=rf"$|\alpha|^2 = {a:g}$")

axes[0].set_yscale("log")
axes[0].set_ylabel("bit-flip rate (GHz)")
axes[0].set_title(r"lines: sweep, $\omega_b = \omega_d$;  stars: compensated_shift",
                  fontsize=9)
axes[1].set_yscale("log")
axes[1].axhline(1.0, color="k", lw=0.8, ls=":")
axes[1].set_ylabel("rate / compensated_shift rate")
axes[1].set_title(rf"$\varepsilon_p$ = {EPS_P:g}", fontsize=9)
for ax in axes:
    ax.axvline(-D_a_mhz, color="0.4", lw=0.8, ls=":")
    ax.annotate(r"$\omega_d/2 - D_a$", (-D_a_mhz, 0), xycoords=("data", "axes fraction"),
                xytext=(3, 4), textcoords="offset points", fontsize=8, color="0.3")
    ax.set_xlabel(r"$(\omega_a - \omega_d/2)\,/\,2\pi$  (MHz)")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(True, which="major", lw=0.4, color="0.9")
    ax.set_axisbelow(True)
axes[1].legend(fontsize=8, frameon=False)
plt.show()

### Against cat size

Left: the uncompensated rate (`shift_a = 0`), `compensated_shift`, and the best
point of each sweep. Right: where that best point sits, as `shift_a` (1 is the
second-order compensation). It is only resolved to the grid spacing, which is
finest between 0.85 and 1.1.

In [ ]:
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
alphas = np.array(list(summary))

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0), constrained_layout=True)
ax = axes[0]
for key, label, c, mk in (("unc", r"uncompensated ($\mathrm{shift}_a = 0$)", BLUE, "s"),
                          ("comp", "compensated_shift", ORANGE, "*"),
                          ("min", "minimum along the sweep", AQUA, "o")):
    ax.plot(alphas, [rate_of(summary[a][key]) for a in alphas], "-", marker=mk,
            ms=7 if mk != "*" else 10, lw=1.1, color=c, label=label)
ax.set_yscale("log")
ax.set_xlabel(r"$|\alpha|^2$")
ax.set_ylabel("bit-flip rate (GHz)")
ax.legend(fontsize=8, frameon=False)

ax = axes[1]
ax.plot(alphas, [summary[a]["min"]["shift_a"] if summary[a]["min"] else np.nan
                 for a in alphas], "o-", ms=7, lw=1.1, color=AQUA)
ax.axhline(1.0, color="k", lw=0.8, ls=":")
ax.set_xlabel(r"$|\alpha|^2$")
ax.set_ylabel(r"$\mathrm{shift}_a$ at the minimum")
ax.set_title(r"$\omega_a = \omega_d/2 - \mathrm{shift}_a\,D_a$", fontsize=9)

for ax in axes:
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(True, which="major", lw=0.4, color="0.9")
    ax.set_axisbelow(True)
plt.show()

### Truncation check

Same criterion as the other notebooks: the top-3 Fock weight of each mode below
`1e-11` at every point. Only the failing points are listed.

In [ ]:
TOL = 1e-11

print(f"{'alpha_sq':>8} {'n_a x n_b':>10} {'max tail_a':>11} {'max tail_b':>11} {'verdict':>8}")
for a, res in results.items():
    ok = [r for r in res if "error" not in r]
    if not ok:
        print(f"{a:8g}  no successful point")
        continue
    failed = [r for r in ok if max(r["tail_a"], r["tail_b"]) >= TOL]
    print(f"{a:8g} {ok[0]['n_a']:4d} x{ok[0]['n_b']:4d} "
          f"{max(r['tail_a'] for r in ok):11.2e} {max(r['tail_b'] for r in ok):11.2e} "
          f"{('PASS' if not failed else 'FAIL'):>8}")
    for r in failed:
        print(f"         {r['label']:<18} tail_a = {r['tail_a']:.2e}  tail_b = {r['tail_b']:.2e}")